<a href="https://colab.research.google.com/github/mahibalavelusamy-ai/MIND-BRIDGE/blob/main/phytoscan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## PhytoScan: AI-Assisted Plant Disease Diagnosis and Guidance

This notebook demonstrates the training of a plant-disease image classifier using TensorFlow/Keras, MobileNetV2 with transfer learning, and the PlantVillage dataset. The final model will be converted to TensorFlow.js format for use in a web application.

### 1. Setup Environment

First, we'll install all necessary libraries, check for GPU availability, and import the required modules.

In [1]:
# Install necessary libraries
!pip install -q tensorflow-datasets tensorflow-io tensorflowjs==4.19.0 scikit-learn

ERROR: Could not find a version that satisfies the requirement tensorflow==2.16.1 (from versions: 2.20.0rc0, 2.20.0, 2.21.0rc0, 2.21.0rc1, 2.21.0, 2.22.0rc0)
ERROR: No matching distribution found for tensorflow==2.16.1


In [2]:
import os
import json
import pathlib
import shutil
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
import tensorflow_datasets as tfds
import tensorflowjs as tfjs # Import tensorflowjs with alias
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print(f"TensorFlow Version: {tf.__version__}")
print(f"Keras Version: {keras.__version__}")
print(f"TensorFlow.js Converter Version: {tfjs.__version__}")

ERROR:absl:Detected incompatible Protobuf Gencode/Runtime versions when loading tensorflow_metadata/proto/v0/anomalies.proto: gencode 6.31.1 runtime 5.29.6. Runtime version cannot be older than the linked gencode version. See Protobuf version guarantees at https://protobuf.dev/support/cross-version-runtime-guarantee.
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/tensorflow_datasets/__init__.py", line 79, in <module>
    from tensorflow_datasets import rlds  # pylint: disable=g-bad-import-order
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/tensorflow_datasets/rlds/__init__.py", line 21, in <module>
    from tensorflow_datasets.rlds import envlogger_reader
  File "/usr/local/lib/python3.13/dist-packages/tensorflow_datasets/rlds/envlogger_reader.py", line 21, in <module>
    from tensorflow_datasets.core.utils.lazy_imports_utils import tree
  File "/usr/local/lib/python3.13/dist-packages/tensorflow_datasets/co

TensorFlow Version: 2.20.0
Keras Version: 3.13.2


NameError: name 'tfjs' is not defined

In [3]:
# Check for GPU availability
physical_devices = tf.config.list_physical_devices('GPU')
if len(physical_devices) > 0:
    print(f"GPU available: {physical_devices[0].name}")
    tf.config.experimental.set_memory_growth(physical_devices[0], True)
else:
    print("No GPU detected. Training will run on CPU, which may be slow.")

No GPU detected. Training will run on CPU, which may be slow.


### 2. Data Loading and Preprocessing

We will use the PlantVillage dataset, which is publicly available. We'll download it, inspect its structure, and then split it into training, validation, and test sets. Finally, we'll apply MobileNetV2-compatible preprocessing and data augmentation.

In [ ]:
# Define image dimensions
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32 # A common batch size for training

# Load the PlantVillage dataset using tensorflow_datasets
# This will download and prepare the dataset if not already present.
print("Downloading and preparing PlantVillage dataset...")
dataset, info = tfds.load('plant_village', with_info=True, as_supervised=True, shuffle_files=True)
print("Dataset loaded successfully!")

### 2.1 Dataset Exploration

Let's inspect the loaded dataset to understand its structure, including the number of classes, class names, and total image count.

In [ ]:
train_ds_raw = dataset['train']

# Get class names
class_names = info.features['label'].names
num_classes = info.features['label'].num_classes

# Calculate total number of images (estimate from train split if no 'all' split exists)
total_images = 0
for _, _ in tfds.as_numpy(train_ds_raw):
    total_images += 1

print(f"Total number of images in the dataset (estimated from 'train' split): {total_images}")
print(f"Number of classes: {num_classes}")
print("Class names:")
for i, name in enumerate(class_names):
    print(f"  {i}: {name}")

### 2.2 Display Sample Images

Let's visualize some sample images from the dataset to get a better understanding of the data.

In [ ]:
plt.figure(figsize=(10, 10))
for i, (image, label) in enumerate(train_ds_raw.take(9)):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(image.numpy().astype("uint8"))
    plt.title(class_names[label.numpy()])
    plt.axis("off")

### 2.3 Split Data into Training, Validation, and Test Sets

We will split the dataset into 70% for training, 15% for validation, and 15% for testing. To prevent data leakage, we'll shuffle the dataset before splitting.

In [ ]:
# Calculate the number of samples for each split
TRAIN_RATIO = 0.7
VALIDATION_RATIO = 0.15
TEST_RATIO = 0.15

DATASET_SIZE = total_images # Using the estimated total_images from the 'train' split

train_size = int(TRAIN_RATIO * DATASET_SIZE)
val_size = int(VALIDATION_RATIO * DATASET_SIZE)
test_size = DATASET_SIZE - train_size - val_size # Ensure all images are accounted for

print(f"Total images: {DATASET_SIZE}")
print(f"Train images: {train_size}")
print(f"Validation images: {val_size}")
print(f"Test images: {test_size}")

# Shuffle the dataset before splitting to prevent data leakage
# Set a buffer size large enough to shuffle the dataset completely
SHUFFLE_BUFFER_SIZE = DATASET_SIZE # Shuffle buffer size should be at least dataset_size
dataset_shuffled = train_ds_raw.shuffle(SHUFFLE_BUFFER_SIZE, seed=42, reshuffle_each_iteration=False)

# Split the dataset
train_ds = dataset_shuffled.take(train_size)
val_ds = dataset_shuffled.skip(train_size).take(val_size)
test_ds = dataset_shuffled.skip(train_size + val_size).take(test_size)

### 2.4 Image Preprocessing and Data Augmentation

We will apply MobileNetV2-compatible preprocessing, resize images to 224x224, and perform data augmentation only on the training set.

In [ ]:
# Preprocessing function for MobileNetV2
def preprocess_image(image, label):
    image = tf.image.resize(image, IMAGE_SIZE) # Resize to 224x224
    image = tf.cast(image, tf.float32) # Cast to float32
    image = tf.keras.applications.mobilenet_v2.preprocess_input(image) # MobileNetV2 specific preprocessing (rescales to [-1, 1])
    return image, label

# Data augmentation for the training set
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
], name="data_augmentation")

def augment_and_preprocess(image, label):
    image, label = preprocess_image(image, label)
    image = data_augmentation(image, training=True) # Apply augmentation only during training
    return image, label

# Apply preprocessing to all datasets
train_ds = train_ds.map(augment_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("Data preprocessing and augmentation applied to datasets.")